In [1]:
import pandas as pd
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "irrigation_prediction.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,Clay,6.14,36.48,0.42,2.17,21.90,31.19,1167.70,4.01,1.97,Wheat,Vegetative,Rabi,Rainfed,Reservoir,4.73,Yes,1.98,South,Low
1,Silt,6.41,50.56,0.38,0.23,36.50,26.01,831.28,10.72,16.82,Maize,Flowering,Zaid,Canal,Groundwater,12.22,Yes,33.56,Central,Medium
2,Sandy,7.71,40.07,1.09,2.18,41.83,76.41,1844.45,7.75,19.03,Cotton,Harvest,Rabi,Drip,Reservoir,5.52,Yes,34.62,South,Low
3,Clay,5.96,12.75,1.56,0.40,37.22,43.32,306.26,8.90,11.44,Wheat,Sowing,Kharif,Canal,Reservoir,1.43,Yes,84.03,North,Medium
4,Clay,7.76,18.58,0.95,2.52,22.38,86.44,1875.63,10.39,11.26,Cotton,Sowing,Zaid,Canal,River,2.52,No,60.86,South,Medium


In [3]:
X = df.drop("Irrigation_Need", axis=1)

y = df["Irrigation_Need"]

In [4]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print(numerical_features)
print(categorical_features)

['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


C:\Users\nudas\AppData\Local\Temp\ipykernel_19816\1397741986.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [6]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

In [7]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

In [9]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
}

## Reflection

### 1. Why do numerical and categorical features require different preprocessing?

Numerical and categorical features represent different types of data and therefore require different preprocessing techniques.

- **Numerical features** contain continuous or discrete numbers. They are often scaled using techniques such as StandardScaler so that features with larger values do not dominate the learning process.
- **Categorical features** contain labels or categories (such as soil type or crop type). Machine learning models cannot directly process text categories, so these features must first be converted into numerical representations using methods like OneHotEncoder.

Applying the correct preprocessing to each feature type helps the model learn more effectively and improves overall performance.

---

### 2. Why are we using Pipeline instead of preprocessing manually?

A `Pipeline` automates the sequence of preprocessing steps and ensures they are applied consistently.

Using a pipeline provides several benefits:
- Keeps the code clean and organized.
- Prevents accidentally applying different preprocessing to training and testing data.
- Reduces the risk of data leakage.
- Makes it easy to reuse the same preprocessing when making predictions on new data.
- Simplifies model training and evaluation.

---

### 3. What advantage does `ColumnTransformer` provide?

`ColumnTransformer` allows different preprocessing techniques to be applied to different groups of columns within the same dataset.

For example:
- Numerical columns can be scaled using `StandardScaler`.
- Categorical columns can be encoded using `OneHotEncoder`.

This eliminates the need to preprocess each subset of columns separately and then manually combine the results. It makes the preprocessing workflow more efficient, readable, and less error-prone.

---

### 4. Why did we use `stratify=y`?

We used `stratify=y` during the train-test split to preserve the class distribution of the target variable in both the training and testing datasets.

This is especially important for classification problems because:
- Both datasets contain approximately the same proportion of each class.
- Model evaluation becomes more reliable.
- It reduces the chance that one dataset contains too many or too few examples of a particular class.

Maintaining the original class distribution helps the model generalize better and provides a fair assessment of its performance.

In [10]:
results = {}

In [11]:
for model_name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", model),
        ]
    )

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted",
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
    )

    results[model_name] = {
        "pipeline": pipeline,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [12]:
results_df = pd.DataFrame({
    model: {
        "Accuracy": values["accuracy"],
        "Precision": values["precision"],
        "Recall": values["recall"],
        "F1 Score": values["f1"],
    }
    for model, values in results.items()
}).T

results_df

,Accuracy,Precision,Recall,F1 Score
Logistic Regression,0.8255,0.824632,0.8255,0.823417
Decision Tree,0.9960,0.996000,0.9960,0.996000
Random Forest,0.9705,0.971181,0.9705,0.966736


In [13]:
results_df = results_df.round(4)

results_df

,Accuracy,Precision,Recall,F1 Score
Logistic Regression,0.8255,0.8246,0.8255,0.8234
Decision Tree,0.9960,0.9960,0.9960,0.9960
Random Forest,0.9705,0.9712,0.9705,0.9667


In [14]:
best_model_name = results_df["Accuracy"].idxmax()

best_model_name

'Decision Tree'

In [15]:
best_pipeline = results[best_model_name]["pipeline"]

In [16]:
print(
    classification_report(
        y_test,
        best_pipeline.predict(X_test),
    )
)

              precision    recall  f1-score   support

        High       0.99      0.99      0.99        67
         Low       1.00      1.00      1.00      1173
      Medium       0.99      0.99      0.99       760

    accuracy                           1.00      2000
   macro avg       0.99      0.99      0.99      2000
weighted avg       1.00      1.00      1.00      2000



In [17]:
PROJECT_ROOT = Path.cwd().parent

MODEL_PATH = PROJECT_ROOT / "models" / "best_model.pkl"

joblib.dump(best_pipeline, MODEL_PATH)

print("Model saved successfully!")

Model saved successfully!


In [18]:
import json

metrics = {
    "Best Model": best_model_name,
    "Accuracy": float(results[best_model_name]["accuracy"]),
    "Precision": float(results[best_model_name]["precision"]),
    "Recall": float(results[best_model_name]["recall"]),
    "F1 Score": float(results[best_model_name]["f1"]),
}

with open(
    PROJECT_ROOT / "reports" / "metrics.json",
    "w",
) as file:
    json.dump(metrics, file, indent=4)

print("Metrics saved successfully!")

Metrics saved successfully!


In [19]:
df.columns.tolist()

['Soil_Type',
 'Soil_pH',
 'Soil_Moisture',
 'Organic_Carbon',
 'Electrical_Conductivity',
 'Temperature_C',
 'Humidity',
 'Rainfall_mm',
 'Sunlight_Hours',
 'Wind_Speed_kmh',
 'Crop_Type',
 'Crop_Growth_Stage',
 'Season',
 'Irrigation_Type',
 'Water_Source',
 'Field_Area_hectare',
 'Mulching_Used',
 'Previous_Irrigation_mm',
 'Region',
 'Irrigation_Need']

Additional Check

In [20]:
from sklearn.model_selection import cross_val_score

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=42)),
    ]
)

scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Cross-validation scores:", scores)
print("Mean accuracy:", scores.mean())

Cross-validation scores: [0.998  0.9945 0.9915 0.997  0.9945]
Mean accuracy: 0.9951000000000001


## Cross-Validation

To verify that the Decision Tree model generalizes well, 5-fold cross-validation was performed.

Mean Cross-Validation Accuracy: **99.51%**

The cross-validation accuracy is very close to the test accuracy (99.60%), indicating that the model performs consistently across different data splits and does not show obvious signs of overfitting.